In [2]:
%pip install pandas pygwalker sqlalchemy pyodbc python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import os
import urllib
from pathlib import Path

import pandas as pd
import pygwalker as pyg
from dotenv import load_dotenv
from sqlalchemy import create_engine

# 1. Carga de entorno y conexión (Formato Dzib)
# Busca .env en la carpeta actual, sus padres y en el árbol del proyecto.
search_roots = []
for base in [Path.cwd(), *Path.cwd().parents, Path.home()]:
    base = base.resolve()
    if base not in search_roots:
        search_roots.append(base)

env_candidates = []
for base in search_roots:
    for candidate in (
        base / ".env",
        base / ".env.local",
        base / ".env.example",
        base / ".env.template",
    ):
        if candidate.is_file():
            env_candidates.append(candidate)

    # Fallback robusto: busca cualquier archivo .env* bajo el árbol del proyecto.
    for candidate in sorted(base.rglob(".env*")):
        if candidate.name.lower() in {".env", ".env.local", ".env.example", ".env.template"}:
            env_candidates.append(candidate)

uniq_candidates = []
seen = set()
for candidate in env_candidates:
    norm = str(candidate.resolve())
    if norm not in seen:
        uniq_candidates.append(candidate)
        seen.add(norm)

env_file = next(
    (candidate for candidate in uniq_candidates if candidate.name.lower() in {".env", ".env.local"}),
    None,
)

if env_file:
    load_dotenv(env_file, override=True)
    print(f"📄 Variables cargadas desde: {env_file}")
else:
    template_file = next(
        (candidate for candidate in uniq_candidates if candidate.name.lower() in {".env.example", ".env.template"}),
        None,
    )
    if template_file:
        print(f"⚠️ No se encontró .env. Usa la plantilla {template_file.name} como referencia.")
    else:
        print("⚠️ No se encontró ningún archivo .env en el proyecto.")
        print("   Crea un archivo .env en la raíz del proyecto con:")
        print("   DB_SERVER=<servidor_sql>")
        print("   DB_NAME=<base_datos>")
        print("   DB_DRIVER=<ODBC Driver 17 for SQL Server>")

server = os.getenv("DB_SERVER")
database = os.getenv("DB_NAME")
driver = os.getenv("DB_DRIVER", "ODBC Driver 17 for SQL Server")

if not server or not database:
    raise RuntimeError(
        "No se encontraron DB_SERVER y/o DB_NAME. "
        "Revisa el archivo .env del proyecto o crea una plantilla basada en .env.example."
    )

params = urllib.parse.quote_plus(
    f"DRIVER={{{driver}}};SERVER={server};DATABASE={database};"
    "Trusted_Connection=yes;TrustServerCertificate=yes;"
)
engine = create_engine(
    f"mssql+pyodbc:///?odbc_connect={params}",
    connect_args={"timeout": 10},
)

# 2. Consumo de Vistas Analíticas
print("📊 Extrayendo datos desde SQL Server...")
df_logistics = pd.read_sql("SELECT * FROM Analytics.vw_Shipping_Efficiency", engine)
df_performance = pd.read_sql("SELECT * FROM Analytics.vw_Category_Performance", engine)

# Validación de contenido (Métrica de éxito: filas recuperadas)
print(f"✅ Filas recuperadas (Logística): {len(df_logistics)}")
print(f"✅ Filas recuperadas (Desempeño): {len(df_performance)}")

# 3. Lanzamiento de Interfaz Interactiva
if not df_logistics.empty:
    print("🚀 Lanzando interfaz interactiva...")
    walker = pyg.walk(df_logistics)
else:
    print("❌ ERROR: El DataFrame de logística está vacío. Revisa la vista en SQL.")


⚠️ No se encontró .env. Usa la plantilla .env.example como referencia.


RuntimeError: No se encontraron DB_SERVER y/o DB_NAME. Revisa el archivo .env del proyecto o crea una plantilla basada en .env.example.